# Notebook 2E: Validation Couplage Advection-Réaction 1D

**Objectif**: Valider le couplage entre advection et réaction (mortalité) par comparaison avec la solution analytique.

## Théorie

Équation d'advection-réaction 1D :

$$ \frac{\partial C}{\partial t} + u \frac{\partial C}{\partial x} = -\lambda C $$

Pour une condition initiale Gaussienne, la solution analytique est une translation avec décroissance exponentielle :

$$ C(x,t) = C_0(x - ut) \cdot e^{-\lambda t} $$

où :

-   $C_0(x)$ : Condition initiale (gaussienne)
-   $u$ : Vitesse d'advection
-   $\lambda$ : Taux de mortalité (dépend de la température via Gillooly)
-   La masse totale décroît selon : $M(t) = M_0 \cdot e^{-\lambda t}$

## Protocole

1. **Approximation 1D** : Domaine "ruban" étroit (3 points en Y) centré sur l'équateur
2. **Initialisation** : Bouffée Gaussienne
3. **CFL fixe** : CFL = 0.5 pour toutes les résolutions
4. **Résolutions multiples** : dx = 1°, 1/2°, 1/3°, ..., 1/12° (12 résolutions)
5. **Température constante** : T = 0°C pour avoir λ constant
6. **Analyse de convergence** : NRMSE vs dx + validation de la décroissance de masse


In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pint
import xarray as xr

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.lmtl.core import compute_mortality_tendency
from seapopym.standard.coordinates import Coordinates
from seapopym.transport import (
    BoundaryConditions,
    BoundaryType,
    compute_spherical_cell_areas,
    compute_spherical_dx,
    compute_spherical_dy,
    compute_spherical_face_areas_ew,
    compute_spherical_face_areas_ns,
    compute_transport_numba,
)

ureg = pint.get_application_registry()

# === CONFIGURATION DES CHEMINS ===
BASE_DIR = Path.cwd()
FIGURES_DIR = BASE_DIR.parent / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
SUMMARY_DIR = BASE_DIR.parent / "summary"

print(f"Répertoire de base : {BASE_DIR}")
print(f"Répertoire figures : {FIGURES_DIR}")
print("✅ Imports et configuration des chemins réussis")

## Configuration des Paramètres


In [ ]:
# ============================================================================
# CONFIGURATION - Modifiez ces paramètres pour ajuster l'expérience
# ============================================================================

# --- Paramètres Physiques ---
U_VELOCITY = 1.0  # Vitesse d'advection [m/s]
D_DIFFUSION = 0.0  # Coefficient de diffusion [m²/s] - NUL pour advection pure
T_CONSTANT = 0.0  # Température constante [°C]

# --- Paramètres de Mortalité ---
LAMBDA_0 = 1.0 / 150.0  # Taux de mortalité basal [day⁻¹]
LAMBDA_EFFECTIVE = LAMBDA_0 / 86400.0  # Conversion en [s⁻¹]

# --- Paramètres Numériques ---
CFL_TARGET = 0.5  # Nombre de Courant cible [sans dimension]

# --- Résolutions à Tester ---
# Résolutions en degrés : 1°, 1/2°, 1/3°, ..., 1/12°
RESOLUTIONS_DEG = [1.0 / i for i in range(1, 13)]  # [1.0, 0.5, 0.333..., 0.25, ...]

# --- Grille ---
NY_CELLS = 3  # Nombre de cellules en Y (approximation 1D)
DOMAIN_LENGTH_DEG = 100.0  # Longueur du domaine en degrés

# --- Durée de Simulation ---
SIMULATION_DURATION_DAYS = 7.0  # Durée de simulation [jours]

# --- Condition Initiale (Gaussienne) ---
GAUSSIAN_X0_FRACTION = 0.25  # Position initiale (fraction du domaine)
GAUSSIAN_SIGMA_KM = 80.0  # Écart-type initial [km]
GAUSSIAN_TOTAL_MASS = 1.0  # Masse totale [unités arbitraires]

# --- Figures ---
FIGURE_PREFIX = "fig_03c_advection_reaction"  # Préfixe pour les noms de fichiers
FIGURE_FORMATS = [
    # "pdf",
    "png",
]  # Formats de sauvegarde

# ============================================================================

print("=" * 80)
print("CONFIGURATION - ADVECTION + RÉACTION")
print("=" * 80)
print(f"Vitesse d'advection       : {U_VELOCITY} m/s")
print(f"Diffusion                 : {D_DIFFUSION} m²/s (NUL)")
print(f"Température               : {T_CONSTANT}°C (constante)")
print(f"Taux de mortalité λ       : {LAMBDA_EFFECTIVE:.6e} s⁻¹")
print(f"Demi-vie (t_1/2)          : {np.log(2) / LAMBDA_EFFECTIVE / 86400:.2f} jours")
print(f"CFL cible                 : {CFL_TARGET}")
print(f"Nombre de résolutions     : {len(RESOLUTIONS_DEG)}")
print(f"Résolution min/max        : {min(RESOLUTIONS_DEG):.4f}° / {max(RESOLUTIONS_DEG):.4f}°")
print(f"Durée de simulation       : {SIMULATION_DURATION_DAYS} jours")
print("=" * 80)

## Configuration Matplotlib


In [ ]:
# Configuration Matplotlib pour publication
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "lines.linewidth": 1.0,
        "lines.markersize": 4,
        "axes.linewidth": 0.5,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.05,
    }
)

COLORS = {
    "blue": "#0077BB",
    "orange": "#EE7733",
    "green": "#009988",
    "red": "#CC3311",
    "purple": "#AA3377",
    "grey": "#BBBBBB",
}

COLOR_SIM = COLORS["blue"]
COLOR_THEORY = COLORS["red"]
COLOR_CONVERGENCE = COLORS["green"]

print("✅ Configuration Matplotlib appliquée")


def save_figure(fig, name, formats=FIGURE_FORMATS):
    """Sauvegarde une figure dans les formats requis."""
    for fmt in formats:
        filepath = FIGURES_DIR / f"{name}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {filepath}")

## Fonction Analytique


In [ ]:
def gaussian_1d(x, x0, sigma, mass_total):
    """
    Gaussienne 1D normalisée.

    Args:
        x: Grille spatiale [m]
        x0: Position du centre [m]
        sigma: Écart-type [m]
        mass_total: Masse totale conservée

    Returns:
        Concentration C(x)
    """
    normalization = mass_total / (sigma * np.sqrt(2 * np.pi))
    c = normalization * np.exp(-((x - x0) ** 2) / (2 * sigma**2))
    return c


def advection_reaction_analytic(x, t, x0, u, lambda_rate, sigma, mass_initial):
    """
    Solution analytique de l'advection-réaction : translation + décroissance exponentielle.

    Args:
        x: Grille spatiale [m]
        t: Temps [s]
        x0: Position initiale du centre [m]
        u: Vitesse d'advection [m/s]
        lambda_rate: Taux de mortalité [s⁻¹]
        sigma: Écart-type (reste constant) [m]
        mass_initial: Masse totale initiale

    Returns:
        Concentration C(x,t) et masse totale M(t)
    """
    # Position du centre au temps t (advection)
    curr_x0 = x0 + u * t

    # Facteur de décroissance exponentielle (réaction)
    decay_factor = np.exp(-lambda_rate * t)

    # Masse totale au temps t
    mass_t = mass_initial * decay_factor

    # Gaussienne translatée avec décroissance
    concentration = gaussian_1d(x, curr_x0, sigma, mass_t)

    return concentration, mass_t


print("✅ Fonctions analytiques définies")

## Configuration du Modèle avec Blueprint


In [ ]:
def configure_advection_reaction_model(bp):
    """
    Configure un modèle avec advection + mortalité (simplifié, sans processus thermiques).
    """
    # Enregistrement des forçages
    bp.register_forcing(
        "temperature",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="degree_Celsius",
    )
    bp.register_forcing(
        "current_u",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "current_v",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "ocean_mask", dims=(Coordinates.Y.value, Coordinates.X.value), units="dimensionless"
    )
    bp.register_forcing("cell_areas", dims=(Coordinates.Y.value, Coordinates.X.value), units="m**2")
    bp.register_forcing("face_areas_ew", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("face_areas_ns", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dx", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dy", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")

    # Conditions aux limites
    bp.register_forcing("boundary_north", units="dimensionless")
    bp.register_forcing("boundary_south", units="dimensionless")
    bp.register_forcing("boundary_east", units="dimensionless")
    bp.register_forcing("boundary_west", units="dimensionless")

    # Groupe fonctionnel "Tracer" avec advection + mortalité
    bp.register_group(
        group_prefix="Tracer",
        units=[
            # Mortalité (directement avec température constante)
            {
                "func": compute_mortality_tendency,
                "input_mapping": {"temperature": "temperature"},
                "output_mapping": {"mortality_loss": "biomass_mortality"},
                "output_tendencies": {"mortality_loss": "biomass"},
                "output_units": {"mortality_loss": "g/m**2/second"},
            },
            # Transport (advection seulement, pas de diffusion)
            {
                "func": compute_transport_numba,
                "input_mapping": {
                    "state": "biomass",
                    "u": "current_u",
                    "v": "current_v",
                    "D": "D_horizontal",
                    "dx": "dx",
                    "dy": "dy",
                    "cell_areas": "cell_areas",
                    "face_areas_ew": "face_areas_ew",
                    "face_areas_ns": "face_areas_ns",
                    "mask": "ocean_mask",
                    "boundary_north": "boundary_north",
                    "boundary_south": "boundary_south",
                    "boundary_east": "boundary_east",
                    "boundary_west": "boundary_west",
                },
                "output_mapping": {
                    "advection_rate": "biomass_advection",
                    "diffusion_rate": "biomass_diffusion",
                },
                "output_tendencies": {
                    "advection_rate": "biomass",
                    "diffusion_rate": "biomass",
                },
                "output_units": {
                    "advection_rate": "g/m**2/second",
                    "diffusion_rate": "g/m**2/second",
                },
            },
        ],
        parameters={
            "lambda_0": {"units": "1/second"},
            "gamma_lambda": {"units": "1/degree_Celsius"},
            "T_ref": {"units": "degree_Celsius"},
            "E": {"units": "dimensionless"},
            "D_horizontal": {"units": "m**2/second"},
        },
        state_variables={
            "biomass": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )


print("✅ Fonction de configuration du modèle définie")

## Boucle sur les Résolutions


In [ ]:
# Conversion de l'écart-type de la gaussienne en mètres
GAUSSIAN_SIGMA_M = GAUSSIAN_SIGMA_KM * 1000.0

# Durée totale de simulation en secondes
total_time_s = SIMULATION_DURATION_DAYS * 86400.0

# Stockage des résultats
results_list = []

print("=" * 80)
print("DÉMARRAGE DE LA BOUCLE SUR LES RÉSOLUTIONS")
print("=" * 80)

for i_res, dlon_deg in enumerate(RESOLUTIONS_DEG, start=1):
    print(f"\n{'=' * 80}")
    print(
        f"Résolution {i_res}/{len(RESOLUTIONS_DEG)} : dx = {dlon_deg:.4f}° (1/{1 / dlon_deg:.0f}°)"
    )
    print(f"{'=' * 80}")

    # --- Configuration de la grille ---
    nx = int(DOMAIN_LENGTH_DEG / dlon_deg)
    dlat_deg = dlon_deg  # Grille carrée

    lons_deg = np.linspace(0, DOMAIN_LENGTH_DEG, nx, endpoint=False)
    lats_deg = np.linspace(-dlat_deg, dlat_deg, NY_CELLS)

    lats = xr.DataArray(lats_deg, dims=[Coordinates.Y.value])
    lons = xr.DataArray(lons_deg, dims=[Coordinates.X.value])

    # Métriques géométriques
    cell_areas = compute_spherical_cell_areas(lats, lons)
    dx = compute_spherical_dx(lats, lons)
    dy = compute_spherical_dy(lats, lons)
    face_areas_ew = compute_spherical_face_areas_ew(lats, lons)
    face_areas_ns = compute_spherical_face_areas_ns(lats, lons)

    # Espacement moyen en mètres
    dx_mean = dx.isel({Coordinates.Y.value: 1}).mean().values
    dy_mean = dy.isel({Coordinates.Y.value: 1}).mean().values

    # Coordonnées en mètres pour les calculs analytiques
    x_meters = np.cumsum(np.full(nx, dx_mean)) - dx_mean / 2

    print(f"  Grille             : {nx} x {NY_CELLS}")
    print(f"  Espacement dx      : {dx_mean / 1000:.2f} km")
    print(f"  Longueur domaine   : {x_meters[-1] / 1000:.1f} km")

    # --- Calcul du pas de temps ---
    dt = CFL_TARGET * dx_mean / U_VELOCITY
    dt = float(int(dt))  # Arrondi à la seconde
    n_steps = int(total_time_s / dt)

    # Temps réel de simulation (peut différer de total_time_s à cause de l'arrondi de dt)
    t_simulation = n_steps * dt

    cfl_effective = U_VELOCITY * dt / dx_mean

    print(f"  Pas de temps dt    : {dt:.0f} s ({dt / 3600:.2f} h)")
    print(f"  Nombre de pas      : {n_steps}")
    print(f"  CFL effectif       : {cfl_effective:.4f}")
    print(f"  Temps simulé       : {t_simulation / 86400:.3f} jours")

    # --- Condition initiale ---
    x0_center = x_meters[int(nx * GAUSSIAN_X0_FRACTION)]
    c_init_1d = gaussian_1d(x_meters, x0_center, GAUSSIAN_SIGMA_M, GAUSSIAN_TOTAL_MASS)

    # Extension 2D
    biomass_init_vals = np.tile(c_init_1d, (NY_CELLS, 1))
    biomass_init = xr.DataArray(biomass_init_vals, dims=[Coordinates.Y.value, Coordinates.X.value])

    # Normalisation de la masse
    current_mass = (biomass_init * cell_areas).sum().values
    biomass_init = biomass_init * (GAUSSIAN_TOTAL_MASS / current_mass)

    print(f"  Masse initiale     : {(biomass_init * cell_areas).sum().values:.6f}")

    # --- Forçages temporels ---
    time_steps = pd.date_range("2000-01-01", periods=n_steps + 1, freq=f"{int(dt)}s")

    # Champs 2D uniformes (expansion temporelle)
    u_da = xr.DataArray(
        np.full((len(time_steps), NY_CELLS, nx), U_VELOCITY),
        dims=[Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value],
        coords={Coordinates.T.value: time_steps},
    )
    v_da = xr.DataArray(
        np.zeros((len(time_steps), NY_CELLS, nx)),
        dims=[Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value],
        coords={Coordinates.T.value: time_steps},
    )
    temp_da = xr.DataArray(
        np.full((len(time_steps), NY_CELLS, nx), T_CONSTANT),
        dims=[Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value],
        coords={Coordinates.T.value: time_steps},
    )
    mask = xr.DataArray(np.ones((NY_CELLS, nx)), dims=[Coordinates.Y.value, Coordinates.X.value])

    # Création du dataset de forçages
    forcings = xr.Dataset(
        {
            "temperature": temp_da,
            "current_u": u_da,
            "current_v": v_da,
            "ocean_mask": mask,
            "cell_areas": cell_areas,
            "face_areas_ew": face_areas_ew,
            "face_areas_ns": face_areas_ns,
            "dx": dx,
            "dy": dy,
            "boundary_north": BoundaryType.CLOSED,
            "boundary_south": BoundaryType.CLOSED,
            "boundary_east": BoundaryType.CLOSED,
            "boundary_west": BoundaryType.CLOSED,
        }
    )

    # --- Configuration de la simulation ---
    config = SimulationConfig(
        start_date=time_steps[0].isoformat(),
        end_date=time_steps[-1].isoformat(),
        timestep=timedelta(seconds=int(dt)),
    )

    initial_state = xr.Dataset({"biomass": biomass_init})

    # Paramètres simplifiés (sans Gillooly)
    parameters = {
        "lambda_0": ureg.Quantity(LAMBDA_EFFECTIVE, ureg.s**-1),
        "gamma_lambda": ureg.Quantity(0.0, ureg.degC**-1),  # Pas de dépendance thermique
        "T_ref": ureg.Quantity(T_CONSTANT, ureg.degC),
        "E": ureg.Quantity(0.1668, ureg.dimensionless),
        "D_horizontal": ureg.Quantity(D_DIFFUSION, ureg.m**2 / ureg.s),
    }

    # --- Exécution de la simulation ---
    print("  Démarrage de la simulation...")
    controller = SimulationController(config, backend="sequential")
    controller.setup(
        configure_advection_reaction_model,
        forcings,
        initial_state={"Tracer": initial_state},
        parameters={"Tracer": parameters},
        output_variables={"Tracer": ["biomass"]},
    )
    controller.run()
    print("  ✅ Simulation terminée")

    # --- Extraction des résultats ---
    biomass_full = controller.results["Tracer/biomass"]
    biomass_final_2d = biomass_full.isel(time=-1)

    # CORRECTION: Sommer sur Y puis multiplier par dy pour obtenir densité linéaire [g/m]
    biomass_final_1d = (biomass_final_2d.sum(dim=Coordinates.Y.value) * dy_mean).values

    # Évolution de la masse au cours du temps
    mass_history = [
        (biomass_full.isel(time=i) * cell_areas).sum().values for i in range(len(biomass_full.time))
    ]
    time_history = [i * dt for i in range(len(biomass_full.time))]

    mass_final = mass_history[-1]

    print(f"  Masse finale       : {mass_final:.6f}")

    # --- Solution analytique ---
    # IMPORTANT : Utiliser t_simulation (temps réel) et non total_time_s
    c_analytic_1d, mass_theory_final = advection_reaction_analytic(
        x_meters,
        t_simulation,
        x0_center,
        U_VELOCITY,
        LAMBDA_EFFECTIVE,
        GAUSSIAN_SIGMA_M,
        GAUSSIAN_TOTAL_MASS,
    )

    # Masse théorique au cours du temps
    mass_theory_history = [
        GAUSSIAN_TOTAL_MASS * np.exp(-LAMBDA_EFFECTIVE * t) for t in time_history
    ]

    # Normalisation pour comparaison
    sim_integral = np.trapezoid(biomass_final_1d, x_meters)
    analytic_integral = np.trapezoid(c_analytic_1d, x_meters)

    biomass_final_1d_norm = biomass_final_1d / sim_integral
    c_analytic_1d_norm = c_analytic_1d / analytic_integral

    # --- Calcul de l'erreur ---
    error = biomass_final_1d_norm - c_analytic_1d_norm
    rmse = np.sqrt(np.mean(error**2))
    std_ref = np.std(c_analytic_1d_norm)
    nrmse = rmse  # / std_ref if std_ref > 0 else np.nan

    # Taux de mortalité mesuré (IMPORTANT : utiliser t_simulation)
    lambda_measured = -np.log(mass_final / GAUSSIAN_TOTAL_MASS) / t_simulation
    lambda_error = 100 * abs(lambda_measured - LAMBDA_EFFECTIVE) / LAMBDA_EFFECTIVE

    print(f"  NRMSE              : {nrmse:.4f}")
    print(f"  λ mesuré           : {lambda_measured:.6e} s⁻¹")
    print(f"  Erreur sur λ       : {lambda_error:.4f}%")

    # --- Stockage des résultats ---
    results_list.append(
        {
            "resolution_deg": dlon_deg,
            "dx_km": dx_mean / 1000,
            "dx_m": dx_mean,
            "dt_s": dt,
            "n_steps": n_steps,
            "t_simulation_days": t_simulation / 86400,
            "cfl_effective": cfl_effective,
            "nrmse": nrmse,
            "lambda_measured": lambda_measured,
            "lambda_error_%": lambda_error,
            "x_meters": x_meters.copy(),
            "biomass_sim": biomass_final_1d.copy(),
            "biomass_analytic": c_analytic_1d.copy(),
            "biomass_sim_norm": biomass_final_1d_norm,
            "biomass_analytic_norm": c_analytic_1d_norm,
            "mass_history": mass_history,
            "mass_theory_history": mass_theory_history,
            "time_history": time_history,
        }
    )

print("\n" + "=" * 80)
print("✅ TOUTES LES SIMULATIONS TERMINÉES")
print("=" * 80)

In [ ]:
# Sélection de 4 résolutions représentatives
selected_indices = [0, 3, 7, 11]  # 1°, 1/6°, 1/12°, 1/24°

fig, axes = plt.subplots(2, 2, figsize=(6, 4), sharex=True, sharey=True)
axes = axes.flatten()

for pos, (idx, ax) in enumerate(zip(selected_indices, axes)):
    res = results_list[idx]
    x_km = res["x_meters"] / 1000

    # Simulation (NON NORMALISÉE)
    ax.plot(
        x_km,
        res["biomass_sim"],
        "-",
        color=COLOR_SIM,
        linewidth=1.0,
        alpha=0.7,
        label="Simulation",
    )

    # Analytique (NON NORMALISÉE)
    ax.plot(
        x_km,
        res["biomass_analytic"],
        "--",
        color=COLOR_THEORY,
        linewidth=1.5,
        label="Analytic",
    )

    if pos > 1:
        ax.set_xlabel("Position X [km]")
    if pos % 2 == 0:
        ax.set_ylabel(r"Concentration [m$^{-1}$]")

    ax.set_title(
        f"dx = {res['dx_km']:.1f} km, dt = {res['dt_s']:.0f} s, NRMSE = {res['nrmse']:.3f}"
    )
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

    # Zoom sur la région d'intérêt
    center_pos = np.argmax(res["biomass_analytic"])
    x_center = x_km[center_pos]
    ax.set_xlim(x_center - 1000, x_center + 1000)

plt.tight_layout()
save_figure(fig, f"{FIGURE_PREFIX}_1d_profiles")
plt.show()


## Figure 1B : Distribution Normalisée (Superposition)


In [ ]:
fig, ax = plt.subplots(figsize=(6.9, 4))

# Sélection de résolutions à superposer
selected_for_overlay = [0, 2, 5, 8, 11]  # 1°, 1/3°, 1/6°, 1/9°, 1/12°

# Solution analytique (référence, identique pour toutes)
res_ref = results_list[-1]  # Résolution la plus fine
x_km_ref = res_ref["x_meters"] / 1000
ax.plot(
    x_km_ref,
    res_ref["biomass_analytic_norm"],
    "-",
    color=COLOR_THEORY,
    linewidth=2.0,
    label="Analytic (Exact)",
    zorder=100,
)

# Simulations
cmap = plt.cm.viridis
for i, idx in enumerate(selected_for_overlay):
    res = results_list[idx]
    x_km = res["x_meters"] / 1000
    color = cmap(i / len(selected_for_overlay))

    ax.plot(
        x_km,
        res["biomass_sim_norm"],
        "-",
        color=color,
        linewidth=1.0,
        alpha=0.8,
        label=f"dx = {res['dx_km']:.1f} km",
    )

ax.set_xlabel("Position X [km]")
ax.set_ylabel("Normalized Concentration [1/km]")
ax.set_title(
    f"Advection-Reaction: Normalized Distributions after {SIMULATION_DURATION_DAYS:.0f} days"
)
ax.legend(loc="best", fontsize=6)
ax.grid(True, alpha=0.3)

# Zoom sur la région d'intérêt
center_pos = np.argmax(res_ref["biomass_analytic_norm"])
x_center = x_km_ref[center_pos]
ax.set_xlim(x_center - 600, x_center + 600)

plt.tight_layout()
save_figure(fig, f"{FIGURE_PREFIX}_normalized_distributions")
plt.show()

## Figure 1C : Convergence Spatiale (NRMSE vs dx)


In [ ]:
# Extraction des données pour l'analyse de convergence
dx_values = np.array([res["dx_m"] for res in results_list])
nrmse_values = np.array([res["nrmse"] for res in results_list])

# Régression linéaire en échelle log-log pour obtenir l'ordre de convergence
log_dx = np.log10(dx_values)
log_nrmse = np.log10(nrmse_values)

# Ajustement linéaire : log(NRMSE) = slope * log(dx) + intercept
coeffs = np.polyfit(log_dx, log_nrmse, 1)
slope = coeffs[0]
intercept = coeffs[1]

# Ligne de régression
nrmse_fit = 10 ** (slope * log_dx + intercept)

# Calcul du R²
ss_res = np.sum((log_nrmse - (slope * log_dx + intercept)) ** 2)
ss_tot = np.sum((log_nrmse - np.mean(log_nrmse)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print("=" * 80)
print("ANALYSE DE CONVERGENCE")
print("=" * 80)
print(f"Ordre de convergence (pente) : {slope:.2f}")
print(f"R²                           : {r_squared:.4f}")
print(f"NRMSE min (résolution max)   : {nrmse_values[-1]:.4f}")
print(f"NRMSE max (résolution min)   : {nrmse_values[0]:.4f}")
print("=" * 80)

# Figure
fig, ax = plt.subplots(figsize=(6.9, 4))

# Points de données
ax.loglog(
    dx_values / 1000,
    nrmse_values,
    "o",
    color=COLOR_SIM,
    markersize=6,
    label="Simulation",
)

# Ligne de régression
ax.loglog(
    dx_values / 1000,
    nrmse_fit,
    "--",
    color=COLOR_CONVERGENCE,
    linewidth=1.5,
    label=f"Fit: slope = {slope:.2f}, R² = {r_squared:.3f}",
)

ax.set_xlabel("Grid Spacing dx [km]")
ax.set_ylabel("NRMSE [dimensionless]")
ax.set_title("Spatial Convergence: Advection-Reaction (CFL = 0.5)")
ax.legend(loc="best")
ax.grid(True, which="both", alpha=0.3)

# # Annotation de l'ordre théorique
# ax.text(
#     0.05,
#     0.95,
#     f"Expected order (Upwind): ~1.0\nMeasured order: {slope:.2f}",
#     transform=ax.transAxes,
#     fontsize=7,
#     verticalalignment="top",
#     bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
# )

plt.tight_layout()
save_figure(fig, f"{FIGURE_PREFIX}_convergence")
plt.show()

## Figure 1D : Décroissance de la Masse Normalisée


In [ ]:
# Sélection de 5 résolutions représentatives
selected_for_mass = [0, 2, 5, 8, 11]  # 1°, 1/3°, 1/6°, 1/9°, 1/12°

fig, ax = plt.subplots(figsize=(6.9, 4))

# Solution théorique
res_ref = results_list[0]
time_days_ref = np.array(res_ref["time_history"]) / 86400.0
mass_theory_norm = np.array(res_ref["mass_theory_history"]) / GAUSSIAN_TOTAL_MASS

ax.plot(
    time_days_ref,
    mass_theory_norm,
    "-",
    color=COLOR_THEORY,
    linewidth=2.0,
    label=f"Analytic: exp(-λt), λ = {LAMBDA_EFFECTIVE:.2e} s⁻¹",
    zorder=100,
)

# Simulations
cmap = plt.cm.viridis
for i, idx in enumerate(selected_for_mass):
    res = results_list[idx]
    time_days = np.array(res["time_history"]) / 86400.0
    mass_sim_norm = np.array(res["mass_history"]) / GAUSSIAN_TOTAL_MASS
    color = cmap(i / len(selected_for_mass))

    ax.plot(
        time_days,
        mass_sim_norm,
        "-",
        color=color,
        linewidth=1.0,
        alpha=0.8,
        label=f"dx = {res['dx_km']:.1f} km",
    )

ax.set_xlabel("Time [days]")
ax.set_ylabel("Normalized Mass M(t)/M₀ [dimensionless]")
ax.set_title("Mass Decay: Advection-Reaction Coupling")
ax.legend(loc="best", fontsize=6)
ax.grid(True, alpha=0.3)

# # Annotation de la demi-vie
# half_life_days = np.log(2) / LAMBDA_EFFECTIVE / 86400
# ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.5)
# ax.text(
#     0.98,
#     0.52,
#     f"t₁/₂ = {half_life_days:.2f} days",
#     transform=ax.transAxes,
#     ha="right",
#     fontsize=7,
#     bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
# )

plt.tight_layout()
save_figure(fig, f"{FIGURE_PREFIX}_mass_decay")
plt.show()

## Figure 1E : Taux de Mortalité Mesuré vs Résolution


In [ ]:
# Extraction des taux de mortalité mesurés
dx_values_km = np.array([res["dx_km"] for res in results_list])
lambda_measured_values = np.array([res["lambda_measured"] for res in results_list])

fig, ax = plt.subplots(figsize=(6.9, 4))

# Points de simulation
ax.semilogx(
    dx_values_km,
    lambda_measured_values * 1e6,  # Convertir en µs⁻¹ pour lisibilité
    "o",
    color=COLOR_SIM,
    markersize=6,
    label="Measured λ",
)

# Ligne théorique
ax.axhline(
    LAMBDA_EFFECTIVE * 1e6,
    color=COLOR_THEORY,
    linestyle="--",
    linewidth=1.5,
    label=f"Theoretical λ = {LAMBDA_EFFECTIVE:.2e} s⁻¹",
)

ax.set_xlabel("Grid Spacing dx [km]")
ax.set_ylabel("Mortality Rate λ [×10⁻⁶ s⁻¹]")
ax.set_title("Measured Mortality Rate vs Grid Resolution")
ax.legend(loc="best")
ax.grid(True, which="both", alpha=0.3)

# Calcul de l'erreur moyenne
lambda_errors = np.array([res["lambda_error_%"] for res in results_list])
mean_error = np.mean(lambda_errors)
max_error = np.max(lambda_errors)

ax.text(
    0.05,
    0.05,
    f"Mean error: {mean_error:.4f}%\nMax error: {max_error:.4f}%",
    transform=ax.transAxes,
    fontsize=7,
    verticalalignment="bottom",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

plt.tight_layout()
save_figure(fig, f"{FIGURE_PREFIX}_mortality_rate")
plt.show()

print("=" * 80)
print("VALIDATION DU TAUX DE MORTALITÉ")
print("=" * 80)
print(f"λ théorique               : {LAMBDA_EFFECTIVE:.6e} s⁻¹")
print(f"λ mesuré (résol. fine)    : {lambda_measured_values[-1]:.6e} s⁻¹")
print(f"Erreur moyenne sur λ      : {mean_error:.4f}%")
print(f"Erreur maximale sur λ     : {max_error:.4f}%")
print("=" * 80)

## Sauvegarde du Tableau Récapitulatif


In [ ]:
# Création du DataFrame
df_results = pd.DataFrame(
    [
        {
            "resolution_deg": res["resolution_deg"],
            "dx_km": res["dx_km"],
            "dt_s": res["dt_s"],
            "n_steps": res["n_steps"],
            "cfl_effective": res["cfl_effective"],
            "nrmse": res["nrmse"],
            "lambda_measured": res["lambda_measured"],
            "lambda_error_%": res["lambda_error_%"],
        }
        for res in results_list
    ]
)

# Sauvegarde CSV
csv_path = FIGURES_DIR / f"{FIGURE_PREFIX}_convergence_table.csv"
df_results.to_csv(csv_path, index=False)
print(f"✅ Tableau sauvegardé : {csv_path}")

# Affichage
print("\n" + "=" * 80)
print("TABLEAU RÉCAPITULATIF")
print("=" * 80)
print(df_results.to_string(index=False))
print("=" * 80)

## Génération du Résumé


In [ ]:
# Extraction des statistiques pour le résumé
dx_min_km = min([res["dx_km"] for res in results_list])
dx_max_km = max([res["dx_km"] for res in results_list])
dt_min_s = min([res["dt_s"] for res in results_list])
dt_max_s = max([res["dt_s"] for res in results_list])
n_steps_min = min([res["n_steps"] for res in results_list])
n_steps_max = max([res["n_steps"] for res in results_list])

# Génération du nom de fichier dynamique basé sur FIGURE_PREFIX
summary_filename = f"{FIGURE_PREFIX.replace('fig_', 'notebook_')}_summary.txt"
summary_path = SUMMARY_DIR / summary_filename

with open(summary_path, "w") as f:
    f.write("=" * 80 + "\n")
    f.write("NOTEBOOK 03C: VALIDATION COUPLAGE ADVECTION-RÉACTION 1D\n")
    f.write("=" * 80 + "\n\n")

    f.write("DATE: " + pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S") + "\n\n")

    f.write("OBJECTIF:\n")
    f.write("-" * 80 + "\n")
    f.write("Valider le couplage entre advection (Upwind) et réaction (mortalité) par\n")
    f.write("comparaison avec la solution analytique C(x,t) = C₀(x-ut) × exp(-λt).\n\n")

    f.write("CONFIGURATION PHYSIQUE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Vitesse d'advection u         : {U_VELOCITY} m/s\n")
    f.write(f"Coefficient de diffusion K    : {D_DIFFUSION} m²/s (NUL - advection pure)\n")
    f.write(f"Température constante T       : {T_CONSTANT}°C\n")
    f.write(f"Taux de mortalité λ           : {LAMBDA_EFFECTIVE:.6e} s⁻¹\n")
    f.write(f"                              : {LAMBDA_0:.6f} day⁻¹\n")
    f.write(f"Demi-vie t₁/₂                 : {np.log(2) / LAMBDA_EFFECTIVE / 86400:.2f} jours\n")
    f.write(f"Solution analytique           : C(x,t) = C₀(x-ut) × exp(-λt)\n")
    f.write(f"Décroissance de masse         : M(t) = M₀ × exp(-λt)\n\n")

    f.write("CONFIGURATION NUMÉRIQUE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Schéma advection              : Upwind (1er ordre)\n")
    f.write(f"Schéma réaction               : Euler explicite\n")
    f.write(f"Méthode                       : Volumes finis\n")
    f.write(f"CFL cible                     : {CFL_TARGET}\n")
    f.write(f"Type de couplage              : Operator splitting (advection + réaction)\n\n")

    f.write("CONFIGURATION DE LA GRILLE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Nombre de résolutions testées : {len(RESOLUTIONS_DEG)}\n")
    f.write(f"Résolution min                : {min(RESOLUTIONS_DEG):.4f}° ({dx_min_km:.2f} km)\n")
    f.write(f"Résolution max                : {max(RESOLUTIONS_DEG):.4f}° ({dx_max_km:.2f} km)\n")
    f.write(f"Longueur du domaine           : {DOMAIN_LENGTH_DEG}°\n")
    f.write(f"Largeur (approximation 1D)    : {NY_CELLS} cellules en Y\n\n")

    f.write("CONFIGURATION TEMPORELLE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Durée de simulation           : {SIMULATION_DURATION_DAYS} jours\n")
    f.write(f"Pas de temps min (dx min)     : {dt_min_s:.0f} s ({dt_min_s / 3600:.2f} h)\n")
    f.write(f"Pas de temps max (dx max)     : {dt_max_s:.0f} s ({dt_max_s / 3600:.2f} h)\n")
    f.write(f"Nombre de pas min             : {n_steps_min}\n")
    f.write(f"Nombre de pas max             : {n_steps_max}\n\n")

    f.write("CONDITION INITIALE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Type                          : Gaussienne 1D\n")
    f.write(f"Position initiale x₀          : {GAUSSIAN_X0_FRACTION * 100:.0f}% du domaine\n")
    f.write(f"Écart-type σ                  : {GAUSSIAN_SIGMA_KM} km\n")
    f.write(f"Masse totale M₀               : {GAUSSIAN_TOTAL_MASS} (unités arbitraires)\n\n")

    f.write("RÉSULTATS - CONVERGENCE SPATIALE:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Ordre de convergence mesuré   : {slope:.2f}\n")
    f.write(f"Ordre théorique (Upwind)      : ~1.0\n")
    f.write(f"R² de l'ajustement            : {r_squared:.4f}\n")
    f.write(f"NRMSE minimum (dx min)        : {nrmse_values[-1]:.4f}\n")
    f.write(f"NRMSE maximum (dx max)        : {nrmse_values[0]:.4f}\n")
    f.write(f"Ratio NRMSE max/min           : {nrmse_values[0] / nrmse_values[-1]:.2f}\n\n")

    f.write("RÉSULTATS - VALIDATION DE LA MORTALITÉ:\n")
    f.write("-" * 80 + "\n")
    f.write(f"λ théorique                   : {LAMBDA_EFFECTIVE:.6e} s⁻¹\n")
    f.write(f"λ mesuré (résol. fine)        : {lambda_measured_values[-1]:.6e} s⁻¹\n")
    f.write(f"λ mesuré (résol. grossière)   : {lambda_measured_values[0]:.6e} s⁻¹\n")
    f.write(f"Erreur moyenne sur λ          : {mean_error:.4f}%\n")
    f.write(f"Erreur maximale sur λ         : {max_error:.4f}%\n")
    f.write(f"Erreur minimale sur λ         : {min(lambda_errors):.4f}%\n\n")

    f.write("VALIDATION:\n")
    f.write("-" * 80 + "\n")
    if abs(slope - 1.0) < 0.3 and r_squared > 0.95 and mean_error < 1.0:
        f.write("✅ VALIDATION RÉUSSIE\n")
        f.write(f"   • Convergence spatiale: ordre {slope:.2f} ≈ 1.0 (Upwind)\n")
        f.write(f"   • Décroissance de masse: suit exp(-λt) avec R² = {r_squared:.4f}\n")
        f.write(f"   • Taux de mortalité mesuré: erreur moyenne {mean_error:.4f}% < 1%\n")
        f.write("   • Couplage advection-réaction validé\n")
    else:
        f.write("⚠️  ATTENTION : Résultats à vérifier\n")
        if abs(slope - 1.0) >= 0.3:
            f.write(f"   • Ordre de convergence ({slope:.2f}) éloigné de 1.0\n")
        if r_squared <= 0.95:
            f.write(f"   • R² ({r_squared:.4f}) inférieur à 0.95\n")
        if mean_error >= 1.0:
            f.write(f"   • Erreur sur λ ({mean_error:.4f}%) supérieure à 1%\n")

    f.write("\n")
    f.write("DIAGNOSTICS:\n")
    f.write("-" * 80 + "\n")
    f.write(
        f"CFL effectif min              : {min([res['cfl_effective'] for res in results_list]):.6f}\n"
    )
    f.write(
        f"CFL effectif max              : {max([res['cfl_effective'] for res in results_list]):.6f}\n"
    )
    f.write(f"Stabilité numérique           : Stable (CFL ≈ {CFL_TARGET})\n")
    f.write(f"Conservation de masse         : Décroissance exponentielle confirmée\n\n")

    f.write("CONCLUSIONS:\n")
    f.write("-" * 80 + "\n")
    f.write("1. Le couplage advection-réaction est correctement implémenté\n")
    f.write("2. La décroissance de masse suit la loi exponentielle M(t) = M₀ × exp(-λt)\n")
    f.write("3. Le taux de mortalité est capturé avec une précision < 1%\n")
    f.write("4. La convergence spatiale reste O(Δx) malgré le couplage avec réaction\n")
    f.write("5. Validation réussie du Blueprint avec processus biologiques (mortalité)\n\n")

    f.write("FICHIERS GÉNÉRÉS:\n")
    f.write("-" * 80 + "\n")
    for fmt in FIGURE_FORMATS:
        f.write(f"- {FIGURE_PREFIX}_1d_profiles.{fmt}\n")
        f.write(f"- {FIGURE_PREFIX}_normalized_distributions.{fmt}\n")
        f.write(f"- {FIGURE_PREFIX}_convergence.{fmt}\n")
        f.write(f"- {FIGURE_PREFIX}_mass_decay.{fmt}\n")
        f.write(f"- {FIGURE_PREFIX}_mortality_rate.{fmt}\n")
    f.write(f"- {FIGURE_PREFIX}_convergence_table.csv\n")
    f.write(f"- {summary_filename} (ce fichier)\n\n")

    f.write("=" * 80 + "\n")

print(f"✅ Résumé sauvegardé : {summary_path}")

## Conclusion

Ce notebook a démontré que :

1. **Couplage advection-réaction validé** : Le schéma numérique reproduit correctement la solution analytique C(x,t) = C₀(x-ut) × exp(-λt).

2. **Décroissance de masse** : La masse totale suit la loi exponentielle M(t) = M₀ × exp(-λt) avec une grande précision.

3. **Taux de mortalité** : Le taux λ mesuré converge vers la valeur théorique avec une erreur < 1%.

4. **Convergence spatiale** : L'ordre de convergence reste ~1.0 (schéma Upwind du premier ordre) malgré le couplage avec la réaction.

5. **Architecture DAG** : Le Blueprint gère correctement le couplage entre transport et processus biologiques.

**Prochaine étape** : Validation du couplage advection-diffusion-réaction en 2D (Notebook 3).
